# RoamAI — embedding pipeline

Populates the `description_embedding` VECTOR(384) column on
destinations and activities using
`sentence-transformers/all-MiniLM-L6-v2`.

Enables semantic search in the MCP server's tools:
- Search destinations by mood/interest
- Search activities matching a natural-language query

Idempotent — only encodes rows where the embedding is NULL, so it's
safe to re-run after adding new destinations or activities.

In [ ]:
%pip install -r /Workspace/Users/rajendrannpriyankaa24@gmail.com/roam-ai/requirements.txt --quiet
dbutils.library.restartPython()

## Config

In [ ]:
import sys
sys.path.insert(0, "/Workspace/Users/rajendrannpriyankaa24@gmail.com/roam-ai")

import lakebase
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

## Load the embedding model

First run downloads ~90 MB of model weights (~30 sec). Cached for
subsequent runs.

In [ ]:
print(f"Loading {EMBEDDING_MODEL_NAME}...")
model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

## Embed destinations

Reads destinations where `description_embedding IS NULL`, encodes
the `description` text into a 384-dim vector, and writes it back.

In [ ]:
pending_destinations = lakebase.run_query("""
    SELECT id, name, description
    FROM destinations
    WHERE description_embedding IS NULL
      AND description IS NOT NULL
      AND LENGTH(description) > 0
""")
print(f"Destinations pending embedding: {len(pending_destinations)}")

for dest in pending_destinations:
    text = dest['description']
    print(f"  Embedding {dest['name']} ({len(text)} chars)...")

    vector = model.encode(text).tolist()
    vector_str = "[" + ",".join(str(x) for x in vector) + "]"

    lakebase.run_write(
        "UPDATE destinations SET description_embedding = %s::vector WHERE id = %s",
        (vector_str, dest['id']),
    )

print(f"\n{len(pending_destinations)} destination(s) embedded.")

## Embed activities

Same pattern for the activities table. The MCP server's
`search_activities` tool uses these embeddings for semantic
retrieval based on interests like "peaceful outdoor time" or
"family-friendly water activity."

In [ ]:
pending_activities = lakebase.run_query("""
    SELECT id, name, description
    FROM activities
    WHERE description_embedding IS NULL
      AND description IS NOT NULL
      AND LENGTH(description) > 0
""")
print(f"Activities pending embedding: {len(pending_activities)}")

for act in pending_activities:
    text = act['description']
    print(f"  Embedding {act['name']} ({len(text)} chars)...")

    vector = model.encode(text).tolist()
    vector_str = "[" + ",".join(str(x) for x in vector) + "]"

    lakebase.run_write(
        "UPDATE activities SET description_embedding = %s::vector WHERE id = %s",
        (vector_str, act['id']),
    )

print(f"\n{len(pending_activities)} activity/activities embedded.")

## Verification — test a semantic search

Encode a natural-language query and find the closest destinations.
If the pipeline worked, semantically-related results should rank
higher than keyword matches would.

In [ ]:
test_query = "tropical island with beaches and hiking"
print(f"Query: {test_query!r}\n")

query_vector = model.encode(test_query).tolist()
query_vector_str = "[" + ",".join(str(x) for x in query_vector) + "]"

results = lakebase.run_query(
    """
    SELECT
        name,
        country,
        1 - (description_embedding <=> %s::vector) AS similarity
    FROM destinations
    WHERE description_embedding IS NOT NULL
    ORDER BY description_embedding <=> %s::vector
    LIMIT 5
    """,
    (query_vector_str, query_vector_str),
)

if results:
    print("Top destination matches:")
    for r in results:
        print(f"  {r['similarity']:.3f}  {r['name']}, {r['country']}")
else:
    print("No embedded destinations yet — run the destination embedding cell first.")

## Summary

At this point:
- Destinations table has 384-dim embeddings for every non-null description
- Activities table has 384-dim embeddings for every non-null description
- HNSW cosine index (created in `sql/03` and `sql/04`) enables fast retrieval
- MCP server tools can now do semantic search over both tables

In [ ]:
counts = lakebase.run_query("""
    SELECT
        'destinations' AS table_name,
        COUNT(*) AS total,
        COUNT(description_embedding) AS embedded
    FROM destinations
    UNION ALL
    SELECT
        'activities',
        COUNT(*),
        COUNT(description_embedding)
    FROM activities
""")
for row in counts:
    print(f"  {row['table_name']:15}  {row['embedded']:3} / {row['total']:3} embedded")